<a href="https://colab.research.google.com/github/MaiChiLe113/triageML/blob/main/data/prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Triage — Step 1: `data/prepare.ipynb`
COS30018 Option D · SOC Triage ML pipeline

**Goal of this notebook:** load CICIDS2017 via the Kaggle dataset
`dhoogla/cicids2017` (Version 2, no-metadata parquet, one file per attack
day/category), verify schema, group raw labels into the 7 project
categories, and write `data/manifest.json`.

**Runs top-to-bottom on Google Colab free tier.**

**Dataset access:** attach `dhoogla/cicids2017` (Version 2) via Colab's
Kaggle **Data Explorer** panel (left sidebar) — do **not** use
`kagglehub.dataset_download(...)`, it returns a 403 in this environment.
Colab prints the local mount path when you attach the dataset; this
notebook discovers that path by searching, it never hardcodes it.

Does NOT split, scale, impute, drop constant/duplicate columns, or train.
Those are later steps (`split.py`, `features.py`, `train.py`).

> STOP GATES in this notebook (by design):
> 1. feature-column count must equal `EXPECTED_N_FEATURES = 77` — anything
>    else halts with the full column list printed.
> 2. every raw label must map to one of the 7 categories — **Heartbleed is
>    expected to fail this and halt the notebook.** That is correct
>    behaviour per the project spec: report it, don't bucket or drop it.
> 3. no flow id / timestamp / IP / port column may survive into the
>    feature set.
>
> If any gate fires, stop and report the printed output — do not edit the
> gate to make it pass.

In [ ]:
# --- Setup: reproducibility ---
# WHY:
# - seeds fixed here so this notebook, split.py, and train.py all see identical splits
# - pyarrow reads the dhoogla no-metadata parquet files; installed defensively in case
#   the Colab base image lacks it (silent no-op if already present)
!pip -q install pyarrow tqdm >/dev/null 2>&1

import os
import json
import random
import hashlib
import glob
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# --- Config ---
# WHY:
# - EXPECTED_N_FEATURES = 77 is locked: dhoogla V2 no-metadata already strips 7 metadata
#   columns + 1 zero-variance column from the original 78-79 column CICIDS2017 schema
# - the label column name is checked against known variants, never assumed silently
# - the 7-class taxonomy is fixed by the project spec; this is the canonical order
EXPECTED_N_FEATURES: int = 77
LABEL_COL_CANDIDATES = ["Label", " Label"]
CATEGORIES = [
    "benign", "dos_ddos", "port_scan", "brute_force",
    "web_attack", "botnet", "infiltration",
]

os.makedirs("data", exist_ok=True)

In [ ]:
# --- Locate the Colab-attached Kaggle dataset ---
# WHY:
# - kagglehub.dataset_download() returns 403 in this environment; do not use it
# - Colab's Kaggle Data Explorer integration mounts the dataset locally instead, but the
#   exact mount path is printed by Colab at attach-time -- it must be discovered here,
#   never hardcoded from memory
def find_parquet_root(search_roots: list) -> str:
    """Search known Kaggle-input roots for the attached dataset's parquet files."""
    candidates = []
    for root in search_roots:
        if os.path.isdir(root):
            candidates.extend(
                glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True)
            )
    if not candidates:
        raise SystemExit(
            "STOP: no .parquet files found under " + ", ".join(search_roots) + ".\n"
            "Attach the dataset first: in Colab, open the left sidebar -> the Kaggle "
            "'Data Explorer' panel -> search 'dhoogla/cicids2017' -> pick Version 2 "
            "(no-metadata) -> Add to notebook. Then re-run this cell.\n"
            "Do not use kagglehub.dataset_download() -- it 403s in this environment."
        )
    root_dir = os.path.commonpath(candidates)
    print(f"found {len(candidates)} parquet file(s), common root: {root_dir}")
    for p in sorted(candidates):
        print(" -", p)
    return root_dir

PARQUET_ROOT = find_parquet_root(["/kaggle/input", "/content/kaggle/input", "/content"])
PARQUET_FILES = sorted(glob.glob(os.path.join(PARQUET_ROOT, "**", "*.parquet"), recursive=True))

In [ ]:
# --- Load parquet files: print shape + columns per file, then concatenate ---
# WHY:
# - dhoogla V2 splits CICIDS2017 into one no-metadata parquet per attack day/category
# - printing each file's shape/columns before concatenation catches a mismatched file
#   immediately, instead of surfacing a confusing error only after concat
def load_all_parquets(paths: list) -> pd.DataFrame:
    """Read every attached parquet file, print shape/columns per file, then concat."""
    frames = []
    for p in tqdm(paths, desc="reading parquet files"):
        f = pd.read_parquet(p)
        f.columns = [str(c).strip() for c in f.columns]
        print(f"\n{os.path.basename(p)}: shape={f.shape}")
        print("  columns:", list(f.columns))
        frames.append(f)
    return pd.concat(frames, ignore_index=True)

df = load_all_parquets(PARQUET_FILES)
print("\nconcatenated shape:", df.shape)

In [ ]:
# --- STOP GATE 1: schema verification ---
# WHY:
# - EXPECTED_N_FEATURES is locked at 77; any mismatch means our schema assumption is
#   wrong and must be reported, never silently reindexed or guessed around
# - the label column name is resolved against known variants only, same reasoning
label_matches = [c for c in df.columns if c in LABEL_COL_CANDIDATES]
if not label_matches:
    print("FULL COLUMN LIST:")
    for i, c in enumerate(df.columns):
        print(f"  {i:>2} {c}")
    raise SystemExit(f"STOP: no label column found among {LABEL_COL_CANDIDATES}.")
LABEL_COL = label_matches[0]

feature_cols = [c for c in df.columns if c != LABEL_COL]
n_feat = len(feature_cols)
print(f"feature columns detected: {n_feat} (expected {EXPECTED_N_FEATURES})")

if n_feat != EXPECTED_N_FEATURES:
    print("\nFULL COLUMN LIST:")
    for i, c in enumerate(df.columns):
        print(f"  {i:>2} {c}")
    raise SystemExit(
        f"STOP: {n_feat} feature cols != {EXPECTED_N_FEATURES}. "
        "Report this list back before continuing."
    )
print("schema OK")

In [ ]:
# --- Raw label distribution ---
# WHY: imbalance is the core modelling challenge; document it before any grouping
raw_counts = df[LABEL_COL].value_counts(dropna=False)
print("raw labels:", raw_counts.shape[0])
print(raw_counts)

In [ ]:
# --- STOP GATE 2: category mapping, halt on first unmapped label ---
# WHY:
# - keyword matching (not exact equality) survives CICIDS2017's en-dash/space/case
#   variants across the different daily capture files
# - order matters: "Web Attack - Brute Force" must hit web_attack before brute_force
# - Heartbleed does not map to any of the 7 buckets; halting here (rather than folding
#   it into dos_ddos or dropping it) is the expected, spec-required behaviour -- report
#   it, don't fix it in this notebook
def to_category(label: str) -> str:
    """Map one raw CICIDS2017 label to one of the 7 project categories, or raise."""
    s = str(label).strip().lower().replace("\u2013", "-").replace("\u2014", "-")
    s = " ".join(s.split())
    if s == "benign":
        return "benign"
    if "web attack" in s or "web-attack" in s:
        return "web_attack"
    if "brute" in s or "patator" in s:
        return "brute_force"
    if "portscan" in s or "port scan" in s:
        return "port_scan"
    if "ddos" in s or "dos" in s:
        return "dos_ddos"
    if "bot" in s:
        return "botnet"
    if "infiltration" in s:
        return "infiltration"
    raise ValueError(label)

def map_categories(frame: pd.DataFrame, label_col: str) -> pd.Series:
    """Map every raw label; halt with the offending label(s) if any are unmapped."""
    mapping, unmapped = {}, []
    for raw in frame[label_col].unique():
        try:
            mapping[raw] = to_category(raw)
        except ValueError:
            unmapped.append(raw)
    if unmapped:
        counts = frame[frame[label_col].isin(unmapped)][label_col].value_counts()
        raise SystemExit(
            "STOP: unmapped label(s) found -- do not bucket or drop silently.\n"
            f"Unmapped labels: {unmapped}\n"
            f"Row counts:\n{counts}"
        )
    return frame[label_col].map(mapping)

df["category"] = map_categories(df, LABEL_COL)
cat_counts = df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
print(cat_counts)

In [ ]:
# --- STOP GATE 3: leakage guard ---
# WHY:
# - dhoogla's no-metadata parquet files claim the 7 metadata columns (flow id, src/dst
#   ip, src/dst port, timestamp) are already stripped -- this asserts that claim rather
#   than trusting it blindly
# - the output contract's `entities` block is always null for CICIDS runs specifically
#   because none of these columns are expected to survive into the feature set
LEAKAGE_PATTERNS = [
    "flow id", "src ip", "source ip", "dst ip", "destination ip",
    "src port", "source port", "dst port", "destination port", "timestamp",
]
feature_cols = [c for c in df.columns if c not in (LABEL_COL, "category")]
leaked = [c for c in feature_cols if any(p in c.strip().lower() for p in LEAKAGE_PATTERNS)]
if leaked:
    raise SystemExit(f"STOP: leakage columns survived cleaning: {leaked}")
print("leakage guard OK: no flow id / timestamp / IP / port columns present")

In [ ]:
# --- Save data/manifest.json ---
# WHY:
# - later steps (split.py, train.py) must read schema/counts from this manifest and
#   never re-derive or guess it
# - sha256 over sorted feature values gives a reproducibility fingerprint for the model
#   card; the iso8601 timestamp records exactly when this snapshot was prepared
def build_manifest(frame: pd.DataFrame, label_col: str, categories: list) -> dict:
    """Assemble the manifest dict: row counts, distributions, columns, hash, timestamp."""
    feats = [c for c in frame.columns if c not in (label_col, "category")]
    h = hashlib.sha256()
    h.update(pd.util.hash_pandas_object(frame[sorted(feats)], index=False).values.tobytes())
    return {
        "source_dataset": "dhoogla/cicids2017 (version 2, no-metadata)",
        "n_rows": int(frame.shape[0]),
        "n_features": len(feats),
        "feature_cols": feats,
        "label_col": label_col,
        "categories": categories,
        "raw_label_distribution": frame[label_col].value_counts().to_dict(),
        "category_distribution": (
            frame["category"].value_counts().reindex(categories, fill_value=0).to_dict()
        ),
        "data_hash": h.hexdigest()[:16],
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }

manifest = build_manifest(df, LABEL_COL, CATEGORIES)
with open("data/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(json.dumps({k: v for k, v in manifest.items() if k != "feature_cols"}, indent=2, default=str))
print("\nwrote data/manifest.json")

## Step 1 output

- `data/manifest.json` — row counts, raw label + category distributions,
  full column list, data hash, generated-at timestamp.

**Run this in Colab (attach `dhoogla/cicids2017` v2 first). Paste back:**
1. the discovered parquet root + per-file shapes/columns
2. the schema-gate result (77 OK, or the full column dump if it stopped)
3. the raw label distribution
4. the category-mapping result — **expected to STOP on `Heartbleed`**; paste
   the exact STOP message
5. any other error

I will not write `split.py` until I see this output.